In [ ]:
!apt-get -qq update
!apt-get -qq install -y mafft

import re, textwrap
from collections import OrderedDict

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Nucleotide MAFFT

In [ ]:
INPUT_FASTA  = "/content/drive/MyDrive//timski/nucleotides.fasta"
CLEAN_FASTA  = "/content/drive/MyDrive/timski/hpv_cleaned.fasta"
OUTPUT_FASTA = "/content/drive/MyDrive/timski/hpv_aligned_mafft.fasta"
LOG_FILE     = "/content/drive/MyDrive/timski/mafft_log.txt"

def read_fasta(path):
    records = OrderedDict()
    header = None
    seq_chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records[header] = "".join(seq_chunks).upper()
                header = line[1:].strip()
                seq_chunks = []
            else:
                seq_chunks.append(line)
        if header is not None:
            records[header] = "".join(seq_chunks).upper()
    return records

def write_fasta(records, path, wrap=80):
    with open(path, "w", encoding="utf-8") as f:
        for h, s in records.items():
            f.write(f">{h}\n")
            for chunk in textwrap.wrap(s, wrap):
                f.write(chunk + "\n")

def clean_records_hpv(records,
                      min_len=7000,       # HPV целосен геном е околу 7900bp, 7000 фаќа near-complete, ги исфрла парцијалните
                      max_n_frac=0.05,    # 0.05 за попрецизен alignment
                      drop_duplicates=True):
    cleaned = OrderedDict()
    seen = set()

    kept = 0
    dropped_short = 0
    dropped_n = 0
    dropped_dup = 0

    # IUPAC ambiguous codes -> претворање во N (R,Y,S,W,K,M,B,D,H,V)
    iupac_amb = set(list("RYSWKMBDHV"))

    for h, s in records.items():
        s = s.upper().replace("U", "T")  # U->T

        # заменување IUPAC ambiguous со N
        s = "".join(("N" if ch in iupac_amb else ch) for ch in s)

        # задржување само A,C,G,T,N (се друго испуштање)
        s = re.sub(r"[^ACGTN]", "", s)

        if len(s) < min_len:
            dropped_short += 1
            continue

        n_frac = (s.count("N") / len(s)) if len(s) else 1.0
        if n_frac > max_n_frac:
            dropped_n += 1
            continue

        if drop_duplicates:
            if s in seen:
                dropped_dup += 1
                continue
            seen.add(s)

        cleaned[h] = s
        kept += 1

    stats = {
        "loaded": len(records),
        "kept": kept,
        "dropped_short": dropped_short,
        "dropped_high_N": dropped_n,
        "dropped_duplicates": dropped_dup
    }
    return cleaned, stats

# Load
records = read_fasta(INPUT_FASTA)

# Clean (HPV-tuned)
cleaned, stats = clean_records_hpv(records, min_len=7000, max_n_frac=0.05, drop_duplicates=True)
write_fasta(cleaned, CLEAN_FASTA)

# 3) Run MAFFT
# --auto: стратегија
# --reorder: подобро читање
# --thread -1: сите јадра
!mafft --auto --reorder --thread -1 "{CLEAN_FASTA}" > "{OUTPUT_FASTA}" 2> "{LOG_FILE}"

print("Aligned FASTA:", OUTPUT_FASTA)
print("MAFFT log:", LOG_FILE)